# Dialforge Sales Acceptance v2.2 — Kaggle

Kaggle-native version of the resilient Dialforge sales benchmark.

Before running, open **Notebook Settings** and set:
- **Accelerator: GPU T4 x2** (preferred; another NVIDIA GPU is acceptable)
- **Internet: On**

Then click **Run All**. The benchmark uses only GPU 0 so results remain comparable to the earlier single-T4 Colab runs.

In [ ]:
# ONE-CLICK DIALFORGE SALES KAGGLE ACCEPTANCE v2.2
import base64, json, os, pathlib, subprocess, sys, time, urllib.request
from IPython.display import HTML, FileLink, display
OWNER='SumamaAhmed69'; REPO='Axemetric-Caller-Beta-Runtime'
BOOTSTRAP_PATH='benchmarks/dialforge_sales_kaggle_bootstrap_v2_2.py'
ROOT=pathlib.Path('/kaggle/working') if pathlib.Path('/kaggle/working').exists() else pathlib.Path.cwd()
BOOTSTRAP=ROOT/'dialforge_sales_kaggle_bootstrap_v2_2.py'
REPORT=ROOT/'dialforge-sales-acceptance'/'dialforge-sales-acceptance.html'
JSON_REPORT=ROOT/'dialforge-sales-acceptance'/'dialforge-sales-acceptance.json'
ZIP_REPORT=ROOT/'dialforge-sales-acceptance-results.zip'
LOG=ROOT/'dialforge-sales-kaggle-v2_2.log'
def api_json(url):
    req=urllib.request.Request(url,headers={'User-Agent':'Dialforge-Sales-Kaggle-v2.2','Accept':'application/vnd.github+json'})
    with urllib.request.urlopen(req,timeout=60) as response: return json.loads(response.read().decode('utf-8'))
print('Dialforge Sales Acceptance launcher: kaggle-v2.2')
print('Checking Kaggle GPU...')
gpu=subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
if gpu.returncode!=0: raise RuntimeError('No NVIDIA GPU. In Kaggle Settings choose Accelerator: GPU T4 x2.')
print(gpu.stdout.strip())
print('Resolving one immutable Dialforge source revision...')
source_sha=api_json(f'https://api.github.com/repos/{OWNER}/{REPO}/commits/main?x={time.time_ns()}')['sha']
print('Pinned source:',source_sha)
item=api_json(f'https://api.github.com/repos/{OWNER}/{REPO}/contents/{BOOTSTRAP_PATH}?ref={source_sha}&x={time.time_ns()}')
if item.get('encoding')!='base64' or not item.get('content'): raise RuntimeError('Could not fetch Kaggle bootstrap. Turn Internet ON in Kaggle Settings.')
payload=base64.b64decode(item['content']); text=payload.decode('utf-8')
required=['DIALFORGE SALES KAGGLE ACCEPTANCE v2.2','dialforge_sales_acceptance_v2_2.py','/kaggle/working']
for marker in required:
    if marker not in text: raise RuntimeError(f'Kaggle bootstrap marker missing: {marker}')
compile(text,str(BOOTSTRAP),'exec'); BOOTSTRAP.write_bytes(payload)
env=os.environ.copy(); env['DIALFORGE_SOURCE_SHA']=source_sha; env['CUDA_VISIBLE_DEVICES']='0'
print('\nStarting Dialforge Sales Acceptance v2.2 on Kaggle with LIVE output...\n')
tail=[]
with LOG.open('w',encoding='utf-8') as log:
    proc=subprocess.Popen([sys.executable,str(BOOTSTRAP)],env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line,end=''); log.write(line); log.flush(); tail.append(line.rstrip('\n'))
        if len(tail)>300: tail=tail[-300:]
    code=proc.wait()
if code!=0:
    print('\n===== KAGGLE FAILURE TAIL =====')
    print('\n'.join(tail[-160:]))
    print('\nFull log:',LOG)
    raise RuntimeError(f'Kaggle sales acceptance stopped with exit code {code}.')
if not REPORT.exists() or not JSON_REPORT.exists() or not ZIP_REPORT.exists(): raise RuntimeError('Benchmark finished without all report files.')
print('\n=== DIALFORGE SALES KAGGLE v2.2 COMPLETE ===')
display(HTML(REPORT.read_text(encoding='utf-8')))
print('\nDownload/share these results:')
display(FileLink(str(ZIP_REPORT))); display(FileLink(str(JSON_REPORT))); display(FileLink(str(LOG)))
